In [7]:
# main tokenizer class 
# this code does not handle any special tokens lie <endofthetext|>, <|start_header_id|> etc, will add them in the regex version of tokenizer.

from assets import merge, cons_freq

class tokenizer:
    
    def __init__(self):
        self.text = ""
        self.vocab_size = 256
        self.verbose = False
        self.merges = {} # will include all the merges (int,int) -> (int)
        self.vocab = {}

    def train(self, text, vocab_size, verbose=False):
        if vocab_size < 256:
            return f"Vocab Size less than required! Recommended Vocab Size {self.vocab_size+1}" # hardcoding vocab_size
        
        self.text = text; self.vocab_size = vocab_size; self.verbose = verbose

        curr_vocab = vocab_size - 256
        vocab = {idx: bytes([idx]) for idx in range(256)}

        # encoding plain text using utf-8 conversion
        enc = text.encode('utf-8')
        tokens = list(enc)

        merges = {}

        # apply bpe algo 
        for i in range(curr_vocab):

            # find out the freq of each pair 
            freq = cons_freq(tokens)

            # find out the most freq occ pair
            pair = max(freq, key = freq.get)

            # replace this pair with a new Token
            idx = 256 + i

            tokens = merge(tokens, pair, idx); merges[pair] = idx # storing the idx of newly replaced pair

            vocab[idx] = vocab[pair[0]] + vocab[pair[1]]
            
            # prints
            if verbose:
                print(f"merge {i+1}/{curr_vocab}: {pair} -> {idx} ({vocab[idx]}) had {freq[pair]} occurrences")
            

        self.merges = merges # needed for encoding 
        self.vocab = vocab # needed for decoding (Reverse form of merges)

        
    # convert (char) to (int)
    def encode(self, text):
        text_bytes = text.encode("utf-8")
        ids = list(text_bytes)

        while(len(ids) >= 2):
            new_token = cons_freq(ids)
            pair = min(new_token, key=lambda p: self.merges.get(p, float("inf")))
            if pair not in self.merges:
                break # nothing else can be merged anymore
            # otherwise let's merge the best pair (lowest merge index)
            idx = self.merges[pair]
            ids = merge(ids, pair, idx)
            
        return ids
    
    # convert  (int) into (string)
    def decode(self, ids):
        text_b = b"".join(self.vocab[idx] for idx in ids)
        text = text_b.decode("utf-8")

        return text


In [8]:
# training tokenizer 

tok = tokenizer()

training_text = """
The quick brown fox jumps over the lazy dog.
Programming in Python is fun.
Machine learning and neural networks are interesting.
"""

tok.train(training_text, vocab_size=300, verbose=True)

merge 1/44: (105, 110) -> 256 (b'in') had 6 occurrences
merge 2/44: (101, 32) -> 257 (b'e ') had 4 occurrences
merge 3/44: (115, 32) -> 258 (b's ') had 3 occurrences
merge 4/44: (46, 10) -> 259 (b'.\n') had 3 occurrences
merge 5/44: (256, 103) -> 260 (b'ing') had 3 occurrences
merge 6/44: (104, 257) -> 261 (b'he ') had 2 occurrences
merge 7/44: (114, 111) -> 262 (b'ro') had 2 occurrences
merge 8/44: (110, 32) -> 263 (b'n ') had 2 occurrences
merge 9/44: (101, 114) -> 264 (b'er') had 2 occurrences
merge 10/44: (114, 97) -> 265 (b'ra') had 2 occurrences
merge 11/44: (260, 32) -> 266 (b'ing ') had 2 occurrences
merge 12/44: (97, 114) -> 267 (b'ar') had 2 occurrences
merge 13/44: (32, 110) -> 268 (b' n') had 2 occurrences
merge 14/44: (268, 101) -> 269 (b' ne') had 2 occurrences
merge 15/44: (10, 84) -> 270 (b'\nT') had 1 occurrences
merge 16/44: (270, 261) -> 271 (b'\nThe ') had 1 occurrences
merge 17/44: (271, 113) -> 272 (b'\nThe q') had 1 occurrences
merge 18/44: (272, 117) -> 273 (b'\

In [13]:
# Now encode some stuff

text = "It's me, Sahil"

ids = tok.encode(text)

print(ids)
print("Number of tokens:", len(ids))

[73, 116, 39, 258, 109, 101, 44, 32, 83, 97, 104, 105, 108]
Number of tokens: 13


In [14]:
decoded = tok.decode(ids)

print(decoded)

It's me, Sahil


In [15]:
print(text == decoded)

True


**$MISSION PASSED**